# Lab 5: Quantum Adders

**Student:** ยามาอัลลุดดีน เจ๊ะหามะ  
**Program:** AISE Cohort 3, College of Computing, PSU Phuket  

Notebook นี้ประกอบด้วย Half Adder, Full Adder และ 2-bit Ripple-Carry Adder โดยใช้ Qiskit Aer simulator ส่วนผล QPU จริงของ Half Adder ถูกบันทึกไว้ในรายงานหลัก

In [ ]:
!pip -q install qiskit qiskit-aer matplotlib pandas

In [ ]:
from itertools import product
import pandas as pd
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

SHOTS = 1024
backend = AerSimulator()

def counts(qc):
    return backend.run(qc, shots=SHOTS).result().get_counts()

## 1. Half Adder

q[0]=a, q[1]=b, q[2]=Sum, q[3]=Carry. ผลลัพธ์แสดงเป็น `Carry Sum b a`.

In [ ]:
def half_adder(a, b):
    qc = QuantumCircuit(4, 4)
    if a: qc.x(0)
    if b: qc.x(1)
    qc.cx(0, 2)
    qc.cx(1, 2)
    qc.ccx(0, 1, 3)
    for q in range(4):
        qc.measure(q, q)
    return qc

half_rows = []
for a, b in product([0, 1], repeat=2):
    result = counts(half_adder(a, b))
    state = max(result, key=result.get)
    expected = ['0000', '0110', '0101', '1011'][(a << 1) | b]
    half_rows.append([a, b, expected, state, result[state], 100*result[state]/SHOTS])

pd.DataFrame(half_rows, columns=['a','b','expected','measured','count','probability_%'])

In [ ]:
qc = half_adder(1, 1)
print(qc.draw('text'))
plot_histogram(counts(qc))

## 2. Full Adder

q[0]=a, q[1]=b, q[2]=Carry-in, q[3]=Sum, q[4]=Carry-out.

In [ ]:
def full_adder(a, b, cin):
    qc = QuantumCircuit(5, 2)
    if a: qc.x(0)
    if b: qc.x(1)
    if cin: qc.x(2)
    qc.cx(0, 3)
    qc.cx(1, 3)
    qc.cx(2, 3)
    qc.ccx(0, 1, 4)
    qc.ccx(0, 2, 4)
    qc.ccx(1, 2, 4)
    qc.measure(3, 0)
    qc.measure(4, 1)
    return qc

full_rows = []
for a, b, cin in product([0, 1], repeat=3):
    result = counts(full_adder(a, b, cin))
    state = max(result, key=result.get)
    expected = f"{int(bool(a*b + a*cin + b*cin))}{a ^ b ^ cin}"
    full_rows.append([a, b, cin, expected, state, result[state], 100*result[state]/SHOTS])

pd.DataFrame(full_rows, columns=['a','b','cin','expected_CoutSum','measured','count','probability_%'])

## 3. 2-bit Ripple-Carry Adder

The output is `Carry_out Sum1 Sum0`.

In [ ]:
def ripple_2bit(a, b):
    qc = QuantumCircuit(8, 3)
    if a & 1: qc.x(0)
    if (a >> 1) & 1: qc.x(1)
    if b & 1: qc.x(2)
    if (b >> 1) & 1: qc.x(3)
    # bit 0
    qc.cx(0, 4); qc.cx(2, 4); qc.ccx(0, 2, 6)
    # bit 1 plus carry from bit 0
    qc.cx(1, 5); qc.cx(3, 5); qc.cx(6, 5)
    qc.ccx(1, 3, 7); qc.ccx(1, 6, 7); qc.ccx(3, 6, 7)
    qc.measure(4, 0); qc.measure(5, 1); qc.measure(7, 2)
    return qc

rca_rows = []
for a, b in [(1, 3), (2, 3)]:
    result = counts(ripple_2bit(a, b))
    state = max(result, key=result.get)
    expected = f"{(a+b) >> 2}{(a+b) & 3:02b}"
    rca_rows.append([f'{a:02b}', f'{b:02b}', expected, state, result[state], 100*result[state]/SHOTS])

pd.DataFrame(rca_rows, columns=['A','B','expected','measured','count','probability_%'])

## 4. IBM Quantum hardware results

Half Adder was run on `ibm_kingston` with 1,024 shots for all four inputs. The measured dominant states were:

| Input | Dominant state | Probability | Job ID |
|---|---|---:|---|
| 00 | 0000 | 93.848% | dc2k8teq4uuc73a5f1tg |
| 01 | 0110 | 92.578% | dc2kd5uq4uuc73a5f920 |
| 10 | 0101 | 92.090% | dc2kf9mq4uuc73a5fcc0 |
| 11 | 1011 | 89.160% | dc2h62anj2vs73dgnoag |

Do not place IBM API keys in this notebook.